

Genome-wide association studies, or GWAS, are used to identify genetic variants associated with traits or diseases.

A GWAS usually tests hundreds of thousands to millions of SNPs across the genome.

For each SNP, we ask:

> Is genetic variation at this SNP statistically associated with the phenotype?

In this tutorial, we walk through a complete GWAS workflow using:

- PLINK for genotype quality control
- PCA for population structure
- GCTA fastGWA for mixed-model association testing
- R for QQ plots and Manhattan plots
- METAL for meta-analysis
- GRM inspection for relatedness checks

The goal is not only to run commands, but to understand why each step matters.

### Overview of the GWAS Pipeline

The complete workflow is:

1. Prepare genotype and phenotype files
2. Perform genotype quality control
3. Remove low-quality SNPs
4. Remove low-quality individuals
5. Check heterozygosity outliers
6. Compute principal components
7. Build a genetic relationship matrix
8. Run GWAS without PC adjustment
9. Run GWAS with PC adjustment
10. Compare QQ plots and Manhattan plots
11. Meta-analyze two GWAS results
12. Inspect heterogeneity
13. Identify the top SNP
14. Inspect relatedness using the GRM



### Why Quality Control is Necessary?

Before running GWAS, we must clean the genotype data.

Poor genotype quality can create false associations.

Common problems include:

- SNPs missing in many individuals
- Individuals missing many genotypes
- Very rare variants with unstable estimates
- SNPs violating Hardy-Weinberg equilibrium
- Sample contamination
- Inbreeding
- Unexpected relatives
- Population stratification

A GWAS is only as reliable as the data used to run it.

### Data Used in This Tutorial

We assume two simulated studies:

```bash
study1.bed
study1.bim
study1.fam

study2.bed
study2.bim
study2.fam

study1_pheno.txt
study2_pheno.txt

study1_covariates.txt
study2_covariates.txt
```


Each study contains:

- around 2,000 individuals
- around 50,000 SNPs
- one phenotype
- covariates including sex, age, and PCs

### PLINK File Formats

PLINK binary genotype data are stored in three files.

| File | Meaning |
|---|---|
| `.bed` | Binary genotype data |
| `.bim` | SNP information |
| `.fam` | Individual information |

The `.bim` file contains SNP-level information:

```text
chromosome  SNP_ID  genetic_distance  base_pair_position  allele1  allele2
```

The `.fam` file contains individual-level information:

```text
FID  IID  father_ID  mother_ID  sex  phenotype
```

The `.bed` file stores the actual genotype matrix in compressed binary format.

### Create Working Directory

```bash
mkdir GWAS
cd GWAS
cp -r /home/abdel/2026/gwas_practical/* .
```

Check files:

```bash
ls
```

Inspect the genotype files:

```bash
head study1.fam
head study1.bim
```

Inspect phenotype and covariate files:

```bash
head study1_pheno.txt
head study1_covariates.txt
```


### Question: Is the Phenotype Quantitative or Case-Control?

Look at the phenotype file:

```bash
head study1_pheno.txt
```

If the phenotype is continuous, such as height, BMI, or simulated trait value, then it is quantitative.

If the phenotype is coded as 0/1 or 1/2 for disease status, then it is case-control.

In this practical, the phenotype is treated as a quantitative phenotype because fastGWA is run using a linear mixed model.

#### Software Used

We use:

| Tool | Purpose |
|---|---|
| PLINK 1.9 | QC and genotype processing |
| GCTA | GRM construction and fastGWA |
| R | Plotting and file preparation |
| qqman | QQ plots and Manhattan plots |
| METAL | Meta-analysis |

# Part 1: Quality Control in PLINK

Quality control is one of the most important parts of GWAS.

The order matters.

We usually clean SNPs first and individuals second.

Why?

If many SNPs are poor quality, individuals may appear to have high missingness just because those SNPs failed. So we first remove bad SNPs, then evaluate individual-level quality.

### Step 1.1: Get an Overview of the Data

Run:

```bash
plink --bfile study1 --freq --out study1_freqs
```

This command:

- reads `study1.bed`, `study1.bim`, and `study1.fam`
- reports number of individuals
- reports number of SNPs
- computes allele frequencies

Output files:

```bash
study1_freqs.frq
study1_freqs.log
```

Inspect:

```bash
head study1_freqs.frq
```

The `.frq` file contains allele frequency information.

Important columns usually include:

| Column | Meaning |
|---|---|
| CHR | Chromosome |
| SNP | SNP ID |
| A1 | Allele 1 |
| A2 | Allele 2 |
| MAF | Minor allele frequency |
| NCHROBS | Number of observed chromosomes |

### Step 1.2: SNP Missingness

Some SNPs fail genotyping in many individuals.

This is measured by SNP missingness.

Run:

```bash
plink --bfile study1 --missing --out study1_miss
```

This creates:

```bash
study1_miss.imiss
study1_miss.lmiss
```

| File | Meaning |
|---|---|
| `.imiss` | Missingness per individual |
| `.lmiss` | Missingness per SNP |

Inspect SNP missingness:

```bash
head study1_miss.lmiss
```

The important column is:

```text
F_MISS
```

If:

```text
F_MISS = 0.05
```

then the SNP is missing in 5% of individuals.

### Remove SNPs With Missingness > 2%

We keep SNPs with call rate at least 98%.

```bash
plink --bfile study1 \
  --geno 0.02 \
  --make-bed \
  --out study1_geno
```

Explanation:

| Flag | Meaning |
|---|---|
| `--bfile study1` | Input PLINK binary dataset |
| `--geno 0.02` | Remove SNPs missing in more than 2% of individuals |
| `--make-bed` | Write new PLINK binary files |
| `--out study1_geno` | Output prefix |

Output:

```bash
study1_geno.bed
study1_geno.bim
study1_geno.fam
```

### Step 1.3: Minor Allele Frequency Filtering

Minor allele frequency, or MAF, is the frequency of the less common allele.

Rare SNPs are difficult to test in small samples because:

- there are few minor allele carriers
- standard errors become large
- genotype errors can have large influence
- power is low

We remove SNPs with MAF below 1%.

```bash
plink --bfile study1_geno \
  --maf 0.01 \
  --make-bed \
  --out study1_maf
```

This removes SNPs with:

```text
MAF < 0.01
```

## Should the MAF Threshold Change in a Larger Study?

Yes, possibly.

In a much larger sample, such as 100,000 or 500,000 individuals, we may have enough power to analyze rarer variants.

For example:

| Sample Size | Reasonable MAF threshold |
|---|---|
| 2,000 | 1% or 5% |
| 50,000 | 0.5% or 1% |
| 500,000 | 0.1% may be possible |

However, rare variant analysis often requires additional care, such as burden tests or sequence-level QC.

### Step 1.4: Hardy-Weinberg Equilibrium

Hardy-Weinberg equilibrium gives expected genotype frequencies under random mating.

For allele frequency:

$$
p
$$

the expected genotype frequencies are:

$$
p^2
$$

$$
2p(1-p)
$$

$$
(1-p)^2
$$

Strong deviation from HWE can indicate:

- genotyping error
- population stratification
- inbreeding
- selection
- true disease association in case-control data

Remove SNPs with HWE p-value below:

$$
1 \times 10^{-6}
$$

```bash
plink --bfile study1_maf \
  --hwe 1e-6 \
  --make-bed \
  --out study1_qc_snps
```

## Why Test HWE Only in Controls for Case-Control Studies?

In case-control studies, a truly disease-associated SNP may deviate from HWE among cases.

If we test HWE in all individuals or in cases, we may accidentally remove true disease signals.

Therefore, for case-control GWAS, HWE filtering is often performed in controls only.

### Step 1.5: Individual Missingness

After SNP QC, we check missingness per individual.

Some individuals may have poor DNA quality and many missing genotypes.

Remove individuals missing more than 2% of genotypes:

```bash
plink --bfile study1_qc_snps \
  --mind 0.02 \
  --make-bed \
  --out study1_qc_mind
```

Explanation:

| Flag | Meaning |
|---|---|
| `--mind 0.02` | Remove individuals with missingness > 2% |

### Step 1.6: Heterozygosity Check

Heterozygosity is the proportion of SNPs where an individual carries two different alleles.

Individuals with unusually high heterozygosity may indicate:

- sample contamination
- DNA mixture
- technical artifacts

Individuals with unusually low heterozygosity may indicate:

- inbreeding
- long runs of homozygosity
- poor genotype calling

Compute heterozygosity:

```bash
plink --bfile study1_qc_mind \
  --het \
  --out study1_het
```

Inspect:

```bash
head study1_het.het
```

Important columns:

| Column | Meaning |
|---|---|
| O(HOM) | Observed homozygous genotypes |
| E(HOM) | Expected homozygous genotypes |
| N(NM) | Number of non-missing genotypes |
| F | Inbreeding coefficient estimate |



# Plot Heterozygosity in R

Switch to R.

```r
setwd("YOUR_WORKING_DIRECTORY")

het <- read.table("study1_het.het", header = TRUE)

het$het_rate <- (het$N.NM. - het$O.HOM.) / het$N.NM.

hist(
  het$het_rate,
  breaks = 50,
  main = "Heterozygosity Rate",
  xlab = "Heterozygosity"
)

upper <- mean(het$het_rate) + 3 * sd(het$het_rate)
lower <- mean(het$het_rate) - 3 * sd(het$het_rate)

abline(v = upper, col = "red", lty = 2)
abline(v = lower, col = "red", lty = 2)
```

# Identify Heterozygosity Outliers

```r
outliers <- het[
  het$het_rate < lower | het$het_rate > upper,
  c("FID", "IID")
]

cat("Heterozygosity outliers:", nrow(outliers), "\n")

write.table(
  outliers,
  "study1_het_outliers.txt",
  row.names = FALSE,
  col.names = FALSE,
  quote = FALSE,
  sep = "\t"
)
```

# Remove Heterozygosity Outliers

Back in the terminal:

```bash
plink --bfile study1_qc_mind \
  --remove study1_het_outliers.txt \
  --make-bed \
  --out study1_qc
```

Now `study1_qc` is the final QC'd Study 1 dataset.

# Step 1.7: Summarize QC Results

Check final SNP and individual counts:

```bash
plink --bfile study1_qc \
  --freq \
  --out study1_qc_freqs
```

You should summarize:

| QC Step | Output Prefix | What Was Removed |
|---|---|---|
| Raw data | study1 | None |
| SNP missingness | study1_geno | SNPs with missingness > 2% |
| MAF filter | study1_maf | SNPs with MAF < 1% |
| HWE filter | study1_qc_snps | SNPs with HWE p < 1e-6 |
| Individual missingness | study1_qc_mind | Individuals with missingness > 2% |
| Heterozygosity | study1_qc | Heterozygosity outliers |

# Repeat QC for Study 2

Now repeat the same commands for Study 2.

# Study 2: Overview

```bash
plink --bfile study2 --freq --out study2_freqs
```

# Study 2: SNP Missingness

```bash
plink --bfile study2 --missing --out study2_miss
```

```bash
head study2_miss.lmiss
```

```bash
plink --bfile study2 \
  --geno 0.02 \
  --make-bed \
  --out study2_geno
```

# Study 2: MAF Filtering

```bash
plink --bfile study2_geno \
  --maf 0.01 \
  --make-bed \
  --out study2_maf
```

# Study 2: HWE Filtering

```bash
plink --bfile study2_maf \
  --hwe 1e-6 \
  --make-bed \
  --out study2_qc_snps
```

# Study 2: Individual Missingness

```bash
plink --bfile study2_qc_snps \
  --mind 0.02 \
  --make-bed \
  --out study2_qc_mind
```

# Study 2: Heterozygosity

```bash
plink --bfile study2_qc_mind \
  --het \
  --out study2_het
```

In R:

```r
het2 <- read.table("study2_het.het", header = TRUE)

het2$het_rate <- (het2$N.NM. - het2$O.HOM.) / het2$N.NM.

hist(
  het2$het_rate,
  breaks = 50,
  main = "Study 2 Heterozygosity Rate",
  xlab = "Heterozygosity"
)

upper2 <- mean(het2$het_rate) + 3 * sd(het2$het_rate)
lower2 <- mean(het2$het_rate) - 3 * sd(het2$het_rate)

abline(v = upper2, col = "red", lty = 2)
abline(v = lower2, col = "red", lty = 2)

outliers2 <- het2[
  het2$het_rate < lower2 | het2$het_rate > upper2,
  c("FID", "IID")
]

cat("Study 2 heterozygosity outliers:", nrow(outliers2), "\n")

write.table(
  outliers2,
  "study2_het_outliers.txt",
  row.names = FALSE,
  col.names = FALSE,
  quote = FALSE,
  sep = "\t"
)
```

Back in terminal:

```bash
plink --bfile study2_qc_mind \
  --remove study2_het_outliers.txt \
  --make-bed \
  --out study2_qc
```

Check final Study 2 data:

```bash
plink --bfile study2_qc \
  --freq \
  --out study2_qc_freqs
```




### QC Interpretation

At the end of QC, we should ask:

1. How many SNPs were removed due to missingness?
2. How many SNPs were removed due to low MAF?
3. How many SNPs were removed due to HWE deviation?
4. How many individuals were removed due to missingness?
5. How many individuals were removed due to heterozygosity outliers?
6. Are the remaining sample sizes reasonable?
7. Are the remaining SNP counts reasonable?

### Why This QC Pipeline Matters

Skipping SNP missingness filtering can leave unreliable variants.

Skipping MAF filtering can produce unstable rare-variant tests.

Skipping HWE filtering can leave genotyping artifacts.

Skipping individual missingness filtering can leave poor-quality samples.

Skipping heterozygosity checks can leave contaminated or inbred samples.

In GWAS, even small QC problems can create genome-wide false positives.


### End of Part 1

At this point we have:

- inspected genotype files
- computed allele frequencies
- removed poorly genotyped SNPs
- removed rare variants
- removed HWE outliers
- removed low-quality individuals
- removed heterozygosity outliers
- produced final QC'd datasets:

```bash
study1_qc
study2_qc
```

Next we will compute principal components to detect and correct population structure.

## Part 2: Principal Components Analysis (PCA) and Population Stratification

#### Why Do We Need PCA?

Suppose we perform a GWAS for a disease.

Imagine:

- Group A has ancestry from Northern Europe
- Group B has ancestry from Southern Europe

Now suppose:

- Disease prevalence differs between groups
- Allele frequencies differ between groups

A SNP can appear associated with disease simply because ancestry differs.

This creates a false positive association.

This phenomenon is called:

### Population Stratification

Population stratification is one of the largest sources of false positives in GWAS.

A GWAS can produce apparently significant associations even when no causal effect exists.

#### Example

Suppose:

| Population | Disease Rate | Allele Frequency |
|------------|-------------|------------------|
| Population A | 10% | 0.20 |
| Population B | 40% | 0.80 |

Even if the SNP has nothing to do with disease:

- cases contain more individuals from Population B
- controls contain more individuals from Population A

The SNP becomes associated with disease.

The association is completely spurious.

# How PCA Helps

Principal Components Analysis identifies major axes of genetic variation.

Instead of looking at:

```text
50,000 SNPs
```

we summarize ancestry using:

```text
PC1
PC2
PC3
...
PC10
```

These PCs can then be included as covariates in the GWAS model.

The PCs absorb ancestry differences.

This dramatically reduces false positives.

# Mathematical Intuition

Suppose our genotype matrix is:

$$
X
$$

with dimensions:

$$
N \times M
$$

where:

- N = individuals
- M = SNPs

For example:

$$
2000 \times 50000
$$

PCA decomposes the genotype matrix into orthogonal directions of variation.

Mathematically:

$$
X = UDV^T
$$

where:

- U contains individual scores
- D contains singular values
- V contains SNP loadings

The first principal component captures the largest source of genetic variation.

Often:

- PC1 reflects ancestry
- PC2 reflects ancestry
- PC3 reflects finer population structure



# Why LD Pruning is Necessary

Before PCA we must remove highly correlated SNPs.

Otherwise:

- large LD blocks dominate the PCA
- certain chromosomes become overrepresented
- ancestry signals become distorted

Therefore we first perform:

## LD Pruning

# Linkage Disequilibrium Refresher

Linkage disequilibrium (LD) measures correlation between nearby SNPs.

For two SNPs:

$$
r^2
$$

measures their correlation.

Values:

| r² | Interpretation |
|-----|--------------|
| 0 | Independent |
| 1 | Perfect correlation |

PCA works best when SNPs are approximately independent.

# Step 2.1 LD Pruning

Run:

```bash
plink --bfile study1_qc \
      --indep-pairwise 50 5 0.2 \
      --out study1_prune
```

# Understanding the Command

```bash
--indep-pairwise 50 5 0.2
```

means:

| Parameter | Meaning |
|------------|---------|
| 50 | Window size |
| 5 | Step size |
| 0.2 | r² threshold |

PLINK:

1. examines 50 SNP windows
2. shifts by 5 SNPs
3. removes SNPs with:

$$
r^2 > 0.2
$$

# Output Files

```bash
study1_prune.prune.in
study1_prune.prune.out
```

### SNPs Retained

```bash
wc -l study1_prune.prune.in
```

These SNPs will be used for PCA.

### SNPs Removed

```bash
wc -l study1_prune.prune.out
```

These SNPs were excluded because of LD.

# Why Not Use All SNPs?

Imagine a chromosome region containing:

```text
500 highly correlated SNPs
```

Without pruning:

- that region contributes 500 times
- another region contributes only once

The PCA becomes biased.

LD pruning gives each genomic region approximately equal influence.

# Step 2.2 Compute Principal Components

Now compute PCs using only pruned SNPs.

```bash
plink --bfile study1_qc \
      --extract study1_prune.prune.in \
      --pca 10 \
      --out study1_pca
```

# Output Files

```bash
study1_pca.eigenvec
study1_pca.eigenval
```

The eigenvectors contain PC scores.

Inspect:

```bash
head study1_pca.eigenvec
```

Example:

```text
FID IID PC1 PC2 PC3 ...
```

Each row represents one individual.

### Variance Explained

Inspect:

```bash
cat study1_pca.eigenval
```

The first eigenvalue corresponds to:

```text
PC1
```

The second eigenvalue corresponds to:

```text
PC2
```

Larger eigenvalues indicate stronger axes of variation.

#### Visualizing Principal Components

Switch to R.

```r
pcs <- read.table(
  "study1_pca.eigenvec",
  header = FALSE
)

colnames(pcs) <- c(
  "FID",
  "IID",
  paste0("PC",1:10)
)
```

# PC1 vs PC2

```r
plot(
  pcs$PC1,
  pcs$PC2,
  xlab="PC1",
  ylab="PC2",
  main="Study 1: PC1 vs PC2",
  pch=20,
  col="steelblue"
)
```

# Interpretation

Each point is one individual.

Individuals close together are genetically similar.

Individuals far apart are genetically different.

# Possible Outcomes

## Scenario 1: Single Cloud

```text
*******
*********
*******
```

Interpretation:

- relatively homogeneous population
- little population structure

## Scenario 2: Two Clusters

```text
****      ****
****      ****
```

Interpretation:

- two ancestry groups
- strong population stratification

## Scenario 3: Gradient

```text
****
  ****
      ****
```

Interpretation:

- continuous ancestry variation
- admixture

#### Real Example

If we analyzed:

- Europeans
- Africans
- East Asians

PC1 and PC2 often separate groups almost perfectly.

The resulting plot contains three distinct clusters.

### Quantifying Variance Explained

Create a scree plot.

```r
eig <- scan(
  "study1_pca.eigenval"
)

var_exp <- eig/sum(eig)

plot(
  var_exp,
  type="b",
  pch=19,
  xlab="Principal Component",
  ylab="Variance Explained",
  main="Scree Plot"
)
```

### Interpretation

Usually:

- PC1 explains most variance
- PC2 explains less
- later PCs explain progressively less

A sharp drop indicates the important ancestry dimensions.

### Visualizing Multiple PCs

PC1 vs PC3:

```r
plot(
  pcs$PC1,
  pcs$PC3,
  pch=20,
  col="darkgreen",
  xlab="PC1",
  ylab="PC3"
)
```

PC2 vs PC3:

```r
plot(
  pcs$PC2,
  pcs$PC3,
  pch=20,
  col="firebrick",
  xlab="PC2",
  ylab="PC3"
)
```

Sometimes structure only appears in later PCs.

### Why PCs Become Covariates

Suppose:

$$
Y
$$

is phenotype.

Instead of fitting:

$$
Y = SNP
$$

we fit:

$$
Y =
SNP +
Age +
Sex +
PC1 +
PC2 +
...
+
PC10
$$

The PCs absorb ancestry effects.

This reduces false associations.

### How Many PCs Should Be Used?

Common choices:

| Dataset | Typical PCs |
|----------|-----------|
| Small GWAS | 5–10 |
| UK Biobank | 10–20 |
| Highly diverse cohort | 20–40 |

There is no universal answer.

Researchers often inspect scree plots and genomic inflation.

# PCA and Relatedness

Close relatives can distort PCA.

For example:

- siblings
- parent-child pairs
- cousins

may create artificial clusters.

Best practice:

1. Identify unrelated individuals.
2. Compute PCs on unrelateds.
3. Project PCs onto relatives.

Tools commonly used:

- PLINK2
- flashPCA
- EIGENSOFT

For this practical we compute PCs directly because the number of relatives is small.

# Repeat PCA for Study 2

Perform the same steps.

LD pruning:

```bash
plink --bfile study2_qc \
      --indep-pairwise 50 5 0.2 \
      --out study2_prune
```

Compute PCs:

```bash
plink --bfile study2_qc \
      --extract study2_prune.prune.in \
      --pca 10 \
      --out study2_pca
```

Plot:

```r
pcs2 <- read.table(
  "study2_pca.eigenvec",
  header=FALSE
)

colnames(pcs2) <- c(
  "FID",
  "IID",
  paste0("PC",1:10)
)

plot(
  pcs2$PC1,
  pcs2$PC2,
  pch=20,
  col="darkorange",
  xlab="PC1",
  ylab="PC2",
  main="Study 2: PC1 vs PC2"
)
```

# Summary

In this section we learned:

1. Population stratification creates false positive GWAS hits.
2. PCA identifies ancestry differences.
3. LD pruning is required before PCA.
4. Principal components summarize genetic variation.
5. PC1 and PC2 often represent ancestry.
6. PCs are added as covariates in GWAS.
7. PCA substantially reduces confounding.

At this point we have:

- QC'd genotype data
- ancestry estimates
- principal components

Next we will build a Genetic Relationship Matrix (GRM) and run a mixed-model GWAS using GCTA fastGWA.

# Part 3: Genetic Relationship Matrices (GRMs) and Mixed-Model GWAS with GCTA fastGWA

# Why Ordinary GWAS Can Fail

Suppose we perform a simple GWAS.

For each SNP we fit:

$$
Y = \beta_0 + \beta_1 SNP + \epsilon
$$

where:

- \(Y\) is the phenotype
- SNP is genotype dosage (0,1,2)
- \(\beta_1\) is the SNP effect

This works well if:

- individuals are unrelated
- there is no population structure

Unfortunately, real cohorts violate both assumptions.

Examples:

- siblings
- cousins
- parent-offspring pairs
- population substructure

These create correlation among observations.

Ordinary regression assumes observations are independent.

Violation of this assumption leads to:

- inflated test statistics
- false positives
- incorrect p-values

# Relatedness Creates Correlated Phenotypes

Imagine two siblings.

They share approximately:

$$
50\%
$$

of their genome.

If a trait has a genetic component, siblings tend to have similar phenotypes.

Their observations are therefore not independent.

A standard GWAS treats them as independent.

This underestimates uncertainty and inflates significance.

# The Solution: Mixed Models

Instead of fitting:

$$
Y = X\beta + \epsilon
$$

we fit:

$$
Y = X\beta + g + \epsilon
$$

where:

- \(X\beta\) = fixed effects
- \(g\) = polygenic random effect
- \(\epsilon\) = residual noise

The random effect captures genetic similarity among individuals.

# What is a Genetic Relationship Matrix?

A Genetic Relationship Matrix (GRM) measures genetic similarity between all pairs of individuals.

Suppose we have:

$$
N
$$

individuals.

The GRM is an:

$$
N \times N
$$

matrix.

Example:

$$
\begin{bmatrix}
1.0 & 0.50 & 0.02\\
0.50 & 1.0 & 0.01\\
0.02 & 0.01 & 1.0
\end{bmatrix}
$$

Interpretation:

- Individual 1 and 2 are siblings
- Individual 3 is unrelated

# Relationship Values

Typical GRM values:

| Relationship | Expected GRM |
|-------------|-------------|
| Same person | 1.0 |
| Parent-child | 0.5 |
| Full siblings | 0.5 |
| Half siblings | 0.25 |
| Grandparent-grandchild | 0.25 |
| First cousins | 0.125 |
| Unrelated | ~0 |

### How the GRM is Computed

Suppose:

$$
x_{ij}
$$

is genotype dosage for SNP \(j\) in individual \(i\).

Genotypes:

| Genotype | Dosage |
|-----------|--------|
| AA | 0 |
| Aa | 1 |
| aa | 2 |

The GRM entry between individuals \(i\) and \(k\) is:

$$
G_{ik}
=
\frac{1}{M}
\sum_{j=1}^{M}
\frac{
(x_{ij}-2p_j)
(x_{kj}-2p_j)
}
{2p_j(1-p_j)}
$$

where:

- \(M\) = number of SNPs
- \(p_j\) = allele frequency

This standardizes each SNP before averaging.

# Visualizing a GRM

Suppose:

```text
Individuals:
A
B
C
D
```

A GRM heatmap might look like:

```text
      A    B    C    D

A   1.0 0.5 0.0 0.0
B   0.5 1.0 0.0 0.0
C   0.0 0.0 1.0 0.2
D   0.0 0.0 0.2 1.0
```

A and B are siblings.

C and D are distant relatives.

# Why Not Use the Full GRM Directly?

Suppose:

$$
N = 500,000
$$

The GRM contains:

$$
500000^2
=
250,000,000,000
$$

entries.

This becomes enormous.

Computational cost grows rapidly.

# Sparse GRMs

fastGWA solves this problem.

Instead of storing every relationship:

```text
0.001
0.002
0.003
```

it stores only meaningful relationships.

Example threshold:

$$
0.05
$$

Any relationship below:

$$
0.05
$$

is replaced by zero.

# Full vs Sparse GRM

Full GRM:

```text
Everyone related to everyone.
```

Sparse GRM:

```text
Only close relatives retained.
```

Advantages:

- smaller memory footprint
- faster computation
- scalable to biobank-sized datasets

# Why PCs Are Still Necessary

This is one of the most important concepts in the workshop.

A sparse GRM captures:

- siblings
- cousins
- close relatives

It does NOT capture:

- ancestry
- population structure

because distant relationships are set to zero.

Therefore:

```text
Sparse GRM
≠
Population Stratification Correction
```

PCs are still required.

# What Corrects What?

| Component | Corrects |
|------------|-----------|
| PCs | Population stratification |
| Sparse GRM | Relatedness |
| Mixed model | Polygenic background |

All are needed.

# Step 3.1 Build the Full GRM

Using GCTA:

```bash
gcta64 \
  --bfile study1_qc \
  --make-grm \
  --out study1_grm \
  --thread-num 4
```

Output:

```text
study1_grm.grm.bin
study1_grm.grm.id
study1_grm.grm.N.bin
```

# What These Files Mean

| File | Purpose |
|--------|----------|
| .grm.bin | GRM values |
| .grm.id | Individual IDs |
| .grm.N.bin | Number of SNPs used |

# Step 3.2 Create Sparse GRM

Convert the full GRM:

```bash
gcta64 \
  --grm study1_grm \
  --make-bK-sparse 0.05 \
  --out study1_sp_grm
```

Threshold:

```text
0.05
```

Relationships below 0.05 become zero.

# Why 0.05?

Approximately:

```text
Closer than third cousins
```

Recommended by GCTA developers.

Balances:

- accuracy
- speed

# Preparing Covariates

GCTA requires:

```text
No header
FID IID covariates
```

# Covariates Without PCs

```r
cov <- read.table(
  "study1_covariates.txt",
  header=TRUE
)

write.table(
  cov[,c(
      "FID",
      "IID",
      "sex",
      "age"
  )],
  "study1_covar_noPCs.txt",
  row.names=FALSE,
  col.names=FALSE,
  quote=FALSE,
  sep="\t"
)
```

# Covariates With PCs

```r
write.table(
  cov[,c(
      "FID",
      "IID",
      "sex",
      "age",
      paste0("PC",1:10)
  )],
  "study1_covar_withPCs.txt",
  row.names=FALSE,
  col.names=FALSE,
  quote=FALSE,
  sep="\t"
)
```

# Prepare Phenotype File

```r
pheno <- read.table(
  "study1_pheno.txt",
  header=TRUE
)

write.table(
  pheno,
  "study1_pheno_gcta.txt",
  row.names=FALSE,
  col.names=FALSE,
  quote=FALSE,
  sep="\t"
)
```

# GWAS Without PC Adjustment

First run the model WITHOUT PCs.

```bash
gcta64 \
 --fastGWA-mlm \
 --bfile study1_qc \
 --grm-sparse study1_sp_grm \
 --pheno study1_pheno_gcta.txt \
 --qcovar study1_covar_noPCs.txt \
 --out study1_noPCs \
 --thread-num 4
```

# Why Do This?

To see the effect of population stratification.

The sparse GRM corrects:

```text
relatedness
```

but not:

```text
ancestry
```

# GWAS With PC Adjustment

Now run the proper model.

```bash
gcta64 \
 --fastGWA-mlm \
 --bfile study1_qc \
 --grm-sparse study1_sp_grm \
 --pheno study1_pheno_gcta.txt \
 --qcovar study1_covar_withPCs.txt \
 --out study1_withPCs \
 --thread-num 4
```

Now we correct:

- relatedness
- ancestry

simultaneously.

# Understanding fastGWA Output

Inspect:

```bash
head study1_withPCs.fastGWA
```

Important columns:

| Column | Meaning |
|----------|---------|
| CHR | Chromosome |
| SNP | SNP ID |
| POS | Base-pair position |
| A1 | Effect allele |
| A2 | Other allele |
| AF1 | Effect allele frequency |
| BETA | Effect size |
| SE | Standard error |
| P | P-value |

# Interpreting Beta

Example:

```text
BETA = 0.25
```

means:

Each additional copy of the effect allele increases the phenotype by 0.25 units.

# Interpreting Standard Error

Smaller:

```text
SE
```

means more precise estimates.

Larger sample sizes generally reduce SE.

# Interpreting P-values

The null hypothesis:

$$
\beta = 0
$$

Small p-values suggest association.

Typical threshold:

$$
5\times10^{-8}
$$

# Why Genome-Wide Significance Is So Stringent

We test:

```text
Hundreds of thousands
to
Millions
of SNPs
```

Multiple testing becomes severe.

Therefore:

$$
5\times10^{-8}
$$

is the accepted threshold.

# Repeat for Study 2

Repeat:

1. Build full GRM
2. Build sparse GRM
3. Create covariates
4. Prepare phenotype file
5. Run fastGWA without PCs
6. Run fastGWA with PCs

Output:

```text
study2_noPCs.fastGWA

study2_withPCs.fastGWA
```

# Summary

In this section we learned:

1. Why ordinary regression fails in related samples.
2. What a GRM measures.
3. How a GRM is computed.
4. Why sparse GRMs are used.
5. Why PCs are still required.
6. How fastGWA works.
7. How to run mixed-model GWAS.
8. How to interpret fastGWA output.

At this point we have completed the association analysis itself.

Next we will visualize the results using:

- QQ plots
- Manhattan plots
- Genomic inflation factor λ

to determine whether our GWAS results are trustworthy.

# Part 4: Visualizing GWAS Results with QQ Plots and Manhattan Plots

# Why Visualization Matters

Running a GWAS produces a table containing thousands or millions of p-values.

A table is difficult to interpret.

Visualization helps answer important questions:

1. Is there population stratification?
2. Is there inflation of test statistics?
3. Are there genuine associations?
4. How many loci reach genome-wide significance?
5. Are significant SNPs clustered in genomic regions?

Two plots dominate GWAS visualization:

- QQ plots
- Manhattan plots

These plots appear in almost every GWAS publication.

# Loading GWAS Results

Switch to R.

```r
library(qqman)

res_noPCs <- read.table(
  "study1_noPCs.fastGWA",
  header = TRUE
)

res_withPCs <- read.table(
  "study1_withPCs.fastGWA",
  header = TRUE
)
```

# Preparing Data for qqman

The qqman package expects:

```text
SNP
CHR
BP
P
```

columns.

Create helper function:

```r
prep <- function(df){

  data.frame(
    SNP = df$SNP,
    CHR = df$CHR,
    BP = df$POS,
    P = df$P
  )

}
```

```r
qq_noPCs <- prep(res_noPCs)

qq_withPCs <- prep(res_withPCs)
```

# Understanding the Null Hypothesis

For every SNP we test:

$$
H_0:\beta=0
$$

Under the null hypothesis:

- SNP has no effect
- p-values follow a Uniform(0,1) distribution

Therefore:

```text
Most p-values should be large.
Few p-values should be small.
```

If the null is true everywhere:

```text
Observed p-values
≈
Expected p-values
```

# What is a QQ Plot?

QQ stands for:

```text
Quantile-Quantile
```

A QQ plot compares:

- observed p-values
- expected p-values under the null

Instead of plotting p-values directly, we use:

$$
-\log_{10}(p)
$$

because small p-values become easier to see.

# Expected Pattern Under the Null

If all SNPs are null:

```text
Observed ≈ Expected
```

The points fall on the diagonal.

Visually:

```text
|
|      /
|     /
|    /
|   /
|__/________
```

# Creating QQ Plots

```r
par(mfrow=c(1,2))

qq(
  qq_noPCs$P,
  main="QQ Plot: No PC Adjustment"
)

qq(
  qq_withPCs$P,
  main="QQ Plot: With PC Adjustment"
)
```

# Interpreting QQ Plots

There are three common patterns.

# Pattern 1: Perfect Null

```text
All points on diagonal
```

Interpretation:

- no inflation
- no true signal

# Pattern 2: Global Inflation

```text
Points rise above diagonal everywhere
```

Interpretation:

Possible causes:

- population stratification
- batch effects
- cryptic relatedness
- poor QC

This is usually bad.

# Pattern 3: Tail Deviation

```text
Most points on diagonal
Only tail rises
```

Interpretation:

- true genetic associations
- well-controlled GWAS

This is the desired pattern.

# Why PC Adjustment Matters

Compare:

```text
No PCs
```

versus

```text
With PCs
```

If PCs successfully correct stratification:

- inflation decreases
- QQ plot approaches diagonal

This demonstrates why PCA was necessary.

# The Genomic Inflation Factor (λ)

A QQ plot is visual.

Lambda provides a numerical summary.

Definition:

$$
\lambda
=
\frac{
\text{median observed } \chi^2
}{
\text{median expected } \chi^2
}
$$

For 1 degree of freedom:

$$
\chi^2_{0.5}
=
0.455
$$

# Computing Lambda

```r
lambda <- function(p){

  chisq <- qchisq(
    1-p,
    df=1
  )

  median(chisq) /
  qchisq(
    0.5,
    df=1
  )

}
```

```r
cat(
  "Lambda (No PCs):",
  lambda(qq_noPCs$P),
  "\n"
)

cat(
  "Lambda (With PCs):",
  lambda(qq_withPCs$P),
  "\n"
)
```

# Interpreting Lambda

| λ | Interpretation |
|----|--------------|
| 1.00 | Ideal |
| 1.02 | Very good |
| 1.05 | Usually acceptable |
| >1.10 | Investigate inflation |
| >1.20 | Likely problematic |

# Important Caveat

Many beginners think:

```text
λ > 1
means
bad GWAS
```

This is not always true.

Large studies often have:

- thousands of real associations
- highly polygenic traits

True signal can also increase λ.

Therefore:

QQ plots should always be interpreted together with λ.

# Manhattan Plots

QQ plots summarize the whole GWAS.

Manhattan plots show where signals occur.

Each SNP is plotted according to:

- genomic position
- significance

# Why the Name "Manhattan"?

Significant loci appear as towers.

These resemble skyscrapers in Manhattan.

# Constructing Manhattan Plots

The x-axis:

```text
Chromosomal position
```

The y-axis:

$$
-\log_{10}(p)
$$

Small p-values become tall peaks.

# Plotting Results

```r
par(mfrow=c(2,1))

manhattan(
  qq_noPCs,
  main="No PC Adjustment",
  suggestiveline=-log10(1e-5),
  genomewideline=-log10(5e-8)
)

manhattan(
  qq_withPCs,
  main="With PC Adjustment",
  suggestiveline=-log10(1e-5),
  genomewideline=-log10(5e-8)
)
```

# Understanding the Threshold Lines

The blue line:

$$
10^{-5}
$$

is the suggestive threshold.

The red line:

$$
5\times10^{-8}
$$

is the genome-wide significance threshold.

# Why 5×10⁻⁸?

Historically:

Approximately one million independent tests occur in a European GWAS.

Using Bonferroni correction:

$$
0.05
/
10^6
=
5\times10^{-8}
$$

This became the standard threshold.

# What Does a True Signal Look Like?

A true causal locus rarely appears as a single SNP.

Instead:

```text
Many nearby SNPs become significant
```

because of linkage disequilibrium.

Result:

```text
Tower
```

rather than:

```text
Single isolated point
```

# Example

Good signal:

```text
      *
     ***
    *****
   *******
```

Suspicious signal:

```text
      *
```

A single isolated SNP often indicates:

- genotyping error
- poor imputation
- technical artifact

# Comparing No-PC and PC-Adjusted Results

Ask:

1. Do significant loci remain?
2. Do some peaks disappear?
3. Does inflation decrease?

Possible outcome:

```text
No PCs:
Many peaks

With PCs:
Fewer peaks
```

Interpretation:

Many initial hits were likely due to stratification.

# Identifying Top Hits

Find the most significant SNPs.

```r
top_hits <- res_withPCs[
  order(res_withPCs$P),
]

head(
  top_hits[
    ,
    c(
      "CHR",
      "SNP",
      "POS",
      "BETA",
      "SE",
      "P"
    )
  ]
)
```

# Volcano Plot (Optional)

Although uncommon in GWAS, a volcano plot can visualize:

- effect size
- significance

```r
plot(
  res_withPCs$BETA,
  -log10(res_withPCs$P),
  pch=20,
  col=rgb(0,0,1,0.3),
  xlab="Beta",
  ylab="-log10(P)",
  main="Volcano Plot"
)
```

# Why Manhattan Plots Are More Popular

Volcano plots ignore:

```text
Genomic position
```

Manhattan plots preserve:

```text
Chromosome
Position
LD structure
```

making them more informative.

# Common GWAS Visualization Mistakes

## Mistake 1

Reporting Manhattan plots without QQ plots.

You cannot assess inflation.

## Mistake 2

Reporting λ without QQ plots.

You lose context.

## Mistake 3

Ignoring isolated significant SNPs.

True loci usually form peaks.

## Mistake 4

Comparing Manhattan plots from different studies without checking sample size.

Larger studies naturally produce stronger signals.

# Study 2 Visualization

Repeat:

```r
res2_noPCs <- read.table(
  "study2_noPCs.fastGWA",
  header=TRUE
)

res2_withPCs <- read.table(
  "study2_withPCs.fastGWA",
  header=TRUE
)
```

Create:

- QQ plots
- λ values
- Manhattan plots

Compare:

- Study 1
- Study 2

before meta-analysis.

# Summary

In this section we learned:

1. Why GWAS visualization is essential.
2. How QQ plots detect inflation.
3. How λ is calculated.
4. Why λ > 1 is not always problematic.
5. How Manhattan plots display genome-wide associations.
6. Why true loci form towers.
7. How PC adjustment affects GWAS results.

At this point we have:

- QC'd data
- principal components
- mixed-model GWAS results
- visualized associations

Next we will combine Study 1 and Study 2 using METAL and perform GWAS meta-analysis.

### Part 5: GWAS Meta-Analysis Using METAL

### Why Meta-Analysis?

Suppose we run GWAS in:

- Study 1 (N = 2,000)
- Study 2 (N = 2,500)

Each study may lack sufficient power to detect small genetic effects.

Many complex traits are highly polygenic.

Most SNP effects are extremely small.

For example:

$$
\beta = 0.02
$$

may require tens of thousands of samples for reliable detection.

Instead of analyzing studies separately, we can combine evidence across studies.

This process is called:

#### Meta-Analysis

### Advantages of Meta-Analysis

Meta-analysis:

- increases sample size
- increases statistical power
- improves effect-size estimation
- identifies consistent signals
- avoids sharing individual-level genotype data

This last point is particularly important.

Many large consortia share only:

- effect sizes
- standard errors
- p-values

rather than raw genotype data.

#### Historical Perspective

Many famous GWAS discoveries were found through meta-analysis.

Examples include:

- GIANT Consortium (height, BMI)
- Psychiatric Genomics Consortium (PGC)
- CARDIoGRAM
- DIAGRAM
- Alzheimer's Disease Genetics Consortium

Modern GWAS often combine:

```text
10
20
50
100+
cohorts
```

##### Fixed-Effect Meta-Analysis

The simplest model assumes:

Every study estimates the same underlying genetic effect.

Suppose:

Study 1 estimates:

$$
\hat{\beta}_1
$$

with standard error:

$$
SE_1
$$

Study 2 estimates:

$$
\hat{\beta}_2
$$

with standard error:

$$
SE_2
$$

#### Inverse Variance Weighting

The combined effect is:

$$
\hat{\beta}_{meta}
=
\frac{
w_1\hat{\beta}_1
+
w_2\hat{\beta}_2
}
{
w_1+w_2
}
$$

where:

$$
w_i
=
\frac{1}{SE_i^2}
$$

Studies with smaller standard errors receive larger weights.

#### Why Larger Studies Get More Weight

Large studies generally have:

- smaller standard errors
- more precise estimates

Therefore:

$$
SE \downarrow
\Rightarrow
Weight \uparrow
$$

This is exactly what we want.

More reliable studies contribute more strongly.

### What Does METAL Do?

METAL is one of the most widely used GWAS meta-analysis programs.

Input:

```text
Summary statistics
```

Output:

```text
Combined summary statistics
```

METAL does not require:

- genotype data
- phenotype data
- individual-level covariates

Only GWAS summary statistics are needed.

#### Preparing GWAS Results

We previously generated:

```text
study1_withPCs.fastGWA

study2_withPCs.fastGWA
```

These files contain:

- SNP
- chromosome
- position
- beta
- standard error
- p-value

#### Inspecting Results

```bash
head study1_withPCs.fastGWA

head study2_withPCs.fastGWA
```

#### Required Columns

METAL typically requires:

| Column | Meaning |
|----------|---------|
| SNP | Variant identifier |
| A1 | Effect allele |
| A2 | Other allele |
| BETA | Effect size |
| SE | Standard error |
| P | P-value |

#### Harmonization

Before meta-analysis:

effect alleles must match.

For example:

Study 1:

```text
A = effect allele
G = reference allele
```

Study 2:

```text
G = effect allele
A = reference allele
```

If not corrected:

effect estimates will point in opposite directions.

This can completely invalidate results.

#### Example

Study 1:

$$
\beta = +0.15
$$

Study 2:

$$
\beta = -0.15
$$

The apparent disagreement may be entirely due to allele coding.

Always harmonize alleles.

#### Creating a METAL Script

Create:

```bash
metal_script.txt
```

# Contents

```text
SCHEME STDERR

MARKER SNP

ALLELE A1 A2

EFFECT BETA

STDERR SE

PVAL P

PROCESS study1_withPCs.fastGWA

PROCESS study2_withPCs.fastGWA

OUTFILE meta_results .

ANALYZE

QUIT
```

# Understanding the Commands

## SCHEME STDERR

Use inverse-variance weighting.

Weights:

$$
w_i=\frac{1}{SE_i^2}
$$

## MARKER

Specifies SNP identifier column.

```text
SNP
```

## EFFECT

Specifies effect-size column.

```text
BETA
```

## STDERR

Specifies standard error column.

```text
SE
```

## PROCESS

Loads a study.

```text
PROCESS study1
PROCESS study2
```

## ANALYZE

Runs the meta-analysis.

# Running METAL

```bash
metal metal_script.txt
```

Output:

```text
meta_results1.tbl
```

# Inspecting Meta-Analysis Results

```bash
head meta_results1.tbl
```

Common columns:

| Column | Meaning |
|----------|---------|
| MarkerName | SNP ID |
| Allele1 | Effect allele |
| Allele2 | Other allele |
| Effect | Combined beta |
| StdErr | Combined SE |
| P-value | Meta-analysis p-value |
| Direction | Sign of effect in each study |

# Understanding Direction

Example:

```text
++
```

Both studies:

```text
positive effect
```

Example:

```text
--
```

Both studies:

```text
negative effect
```

Example:

```text
+-
```

Studies disagree.

This may indicate:

- heterogeneity
- allele issues
- random variation

# Why Meta-Analysis Increases Power

Suppose:

Study 1:

$$
P=10^{-4}
$$

Study 2:

$$
P=10^{-3}
$$

Neither reaches:

$$
5\times10^{-8}
$$

alone.

Combined analysis may produce:

$$
P=10^{-9}
$$

This is why meta-analysis dominates modern GWAS.

# Comparing GWAS and Meta-Analysis

Load results in R.

```r
study1 <- read.table(
  "study1_withPCs.fastGWA",
  header=TRUE
)

study2 <- read.table(
  "study2_withPCs.fastGWA",
  header=TRUE
)

meta <- read.table(
  "meta_results1.tbl",
  header=TRUE
)
```

# Number of Significant Hits

```r
sum(
  study1$P < 5e-8
)

sum(
  study2$P < 5e-8
)

sum(
  meta$P.value < 5e-8
)
```

# Interpretation

Typically:

```text
Meta-analysis
>
Individual studies
```

in terms of discoveries.

# Manhattan Plot of Meta-Analysis

Prepare data.

```r
library(qqman)

meta_plot <- data.frame(
  SNP = meta$MarkerName,
  CHR = meta$Chromosome,
  BP = meta$Position,
  P = meta$P.value
)
```

```r
manhattan(
  meta_plot,
  main="Meta-analysis Manhattan Plot",
  suggestiveline=-log10(1e-5),
  genomewideline=-log10(5e-8)
)
```

# QQ Plot of Meta-Analysis

```r
qq(
  meta$P.value,
  main="Meta-analysis QQ Plot"
)
```

# Heterogeneity

One of the most important concepts in meta-analysis is:

## Heterogeneity

Suppose:

Study 1:

$$
\beta = 0.20
$$

Study 2:

$$
\beta = 0.18
$$

These are consistent.

Now suppose:

Study 1:

$$
\beta = 0.25
$$

Study 2:

$$
\beta = -0.20
$$

These are inconsistent.

This is heterogeneity.

# Cochran's Q Statistic

Many meta-analysis tools evaluate:

$$
Q
=
\sum
w_i
(
\beta_i-\beta_{meta}
)^2
$$

Large Q values indicate disagreement among studies.

# I² Statistic

Another common metric:

$$
I^2
=
100\%
\times
\frac{Q-df}{Q}
$$

Interpretation:

| I² | Meaning |
|-----|---------|
| 0% | No heterogeneity |
| 25% | Low |
| 50% | Moderate |
| 75% | High |

# Why Heterogeneity Matters

Heterogeneity may arise from:

- ancestry differences
- environmental differences
- phenotype definitions
- technical differences
- gene-environment interaction

# Mystery Phenotype Investigation

The workshop asks:

> Can we identify the phenotype from the strongest GWAS hit?

Procedure:

1. Find the top SNP.

```r
top_snp <- meta[
  which.min(meta$P.value),
]

top_snp
```

2. Record:

```text
rsID
```

3. Search:

- GWAS Catalog
- Ensembl
- dbSNP
- Open Targets Genetics

4. Compare previously reported traits.

This often provides clues regarding phenotype identity.

# Example Resources

Useful databases:

- :contentReference[oaicite:0]{index=0}
- :contentReference[oaicite:1]{index=1}
- :contentReference[oaicite:2]{index=2}

# Summary

In this section we learned:

1. Why GWAS meta-analysis is necessary.
2. How inverse-variance weighting works.
3. How METAL combines studies.
4. Why allele harmonization matters.
5. How to interpret meta-analysis results.
6. How heterogeneity is assessed.
7. Why meta-analysis increases power.
8. How to identify candidate phenotypes using top SNPs.

At this point we have completed:

- QC
- PCA
- GRM construction
- fastGWA analysis
- QQ plots
- Manhattan plots
- Meta-analysis

The final section will focus on:

- inspecting relatedness directly from the GRM
- identifying relatives
- interpreting GRM values
- understanding cryptic relatedness
- best practices for large-scale GWAS.

### Part 6: Understanding Relatedness, GRMs, and Best Practices for GWAS

#### Why Relatedness Matters

One of the fundamental assumptions of classical statistical tests is:

> Observations are independent.

In genetic studies this assumption is often violated.

Individuals may be:

- siblings
- parent-child pairs
- cousins
- twins
- members of the same pedigree

These relationships introduce correlation.

If ignored, GWAS statistics become inflated.

#### What is Cryptic Relatedness?

Cryptic relatedness means:

> Individuals are genetically related, but the relationship is unknown or unrecorded.

Examples:

- unknown cousins
- undocumented family relationships
- pedigree errors
- duplicate samples

Large biobanks often contain thousands of related individuals.

#### Historical Perspective

Early GWAS often removed relatives.

Modern GWAS typically retain relatives and use mixed models.

Why?

Because removing relatives wastes valuable samples.

Mixed models allow us to:

- retain individuals
- model relatedness directly
- increase statistical power

#### Revisiting the Genetic Relationship Matrix

The GRM contains pairwise genetic similarity.

Suppose we have:

$$
N
$$

individuals.

The GRM contains:

$$
N\times N
$$

entries.

Each cell represents:

```text
How genetically similar are two individuals?
```

#### GRM Interpretation

Suppose:

```text
ID1 ID2 Relationship
```

produces:

```text
0.50
```

Interpretation:

Likely:

- parent-child
- full siblings

Suppose:

```text
0.25
```

Interpretation:

Likely:

- half siblings
- grandparent-grandchild
- avuncular relationships

Suppose:

```text
0.125
```

Interpretation:

Likely:

- first cousins

Suppose:

```text
0.00
```

Interpretation:

Essentially unrelated.

#### Typical Relatedness Thresholds

| GRM Value | Interpretation |
|------------|---------------|
| >0.95 | Duplicate sample / identical twin |
| ~0.50 | Parent-child or sibling |
| ~0.25 | Second-degree relative |
| ~0.125 | First cousin |
| ~0 | Unrelated |

These values are approximate.

Real data fluctuate around expectations.

#### Reading the GRM

We previously generated:

```bash
study1_grm
```

using:

```bash
gcta64 \
  --bfile study1_qc \
  --make-grm \
  --out study1_grm
```

Convert GRM to text:

```bash
gcta64 \
  --grm study1_grm \
  --grm-cutoff 0 \
  --make-grm-gz \
  --out study1_grm_text
```

This produces:

```text
study1_grm_text.grm.gz
```

Inspect:

```bash
zcat study1_grm_text.grm.gz | head
```

#### Understanding GRM Columns

Typical output:

```text
ID1
ID2
N_SNP
GRM
```

Meaning:

| Column | Description |
|----------|-------------|
| ID1 | Individual 1 |
| ID2 | Individual 2 |
| N_SNP | Number of SNPs used |
| GRM | Relatedness estimate |

#### Finding Close Relatives

Extract individuals with:

$$
GRM > 0.05
$$

```bash
zcat study1_grm_text.grm.gz \
| awk '$4 > 0.05'
```

These are genetically related pairs.

#### Counting Related Pairs

```bash
zcat study1_grm_text.grm.gz \
| awk '$4 > 0.05' \
| wc -l
```

This gives:

```text
Number of related pairs
```

#### Identifying Duplicates

Potential duplicates:

```bash
zcat study1_grm_text.grm.gz \
| awk '$4 > 0.95'
```

Interpretation:

Possible:

- duplicated sample
- monozygotic twins
- sample labeling error

These should be investigated.

#### Visualizing the GRM

Switch to R.

```r
library(data.table)

grm <- fread(
  "study1_grm_text.grm.gz"
)
```

#### Heatmap of Relatedness

```r
library(ggplot2)

ggplot(
  grm,
  aes(
    x=V1,
    y=V2,
    fill=V4
  )
)+
geom_tile()+
scale_fill_gradient(
  low="white",
  high="red"
)+
theme_minimal()+
labs(
  title="Genetic Relationship Matrix"
)
```

#### Interpretation

Bright red regions indicate:

- families
- clusters of relatives

White regions indicate:

- unrelated individuals

#### Relatedness Distribution

```r
hist(
  grm$V4,
  breaks=100,
  main="GRM Distribution",
  xlab="Relatedness"
)
```

#### Expected Pattern

Most individuals should be:

```text
Near zero
```

A small number should show:

```text
0.125
0.25
0.5
```

indicating relatives.

### Relationship Categories

Create categories.

```r
grm$relationship <- cut(
  grm$V4,
  breaks=c(
    -Inf,
    0.05,
    0.125,
    0.25,
    0.5,
    Inf
  ),
  labels=c(
    "Unrelated",
    "Distant",
    "First Cousin",
    "Second Degree",
    "First Degree"
  )
)

table(
  grm$relationship
)
```

#### Why Relatedness Creates False Positives

Suppose:

- siblings share 50% of genome
- siblings share environmental exposures

Phenotypes become correlated.

A SNP inherited within families may appear associated with the trait simply because family members resemble each other.

This creates inflation.

### Classical Solution

Older GWAS often removed relatives.

Example:

```bash
plink \
  --bfile study1_qc \
  --rel-cutoff 0.125 \
  --make-bed \
  --out study1_unrelated
```

This removes one individual from each related pair.

# Disadvantages

You lose data.

Example:

```text
20,000 samples
```

may become:

```text
14,000 samples
```

after removing relatives.

Power decreases.

# Modern Solution

Use:

```text
Linear Mixed Models
```

Examples:

- fastGWA
- BOLT-LMM
- SAIGE
- REGENIE

These methods:

- keep relatives
- model relatedness directly

This is why modern biobanks rarely remove all relatives.

# Comparing Approaches

| Approach | Relatives Removed? |
|-----------|-------------------|
| PLINK Linear Regression | Usually yes |
| Mixed Models | No |
| fastGWA | No |
| BOLT-LMM | No |
| REGENIE | No |

# Why Biobanks Need Mixed Models

Consider:

### UK Biobank

Approximately:

```text
500,000 individuals
```

Contains:

```text
Tens of thousands
of related individuals
```

Removing all relatives would waste enormous amounts of data.

Mixed models solve this problem.

# GWAS Best Practices Checklist

Before running GWAS:

## Sample QC

✔ Missingness

✔ Sex checks

✔ Heterozygosity

✔ Relatedness

## SNP QC

✔ Missingness

✔ MAF filtering

✔ HWE filtering

## Population Structure

✔ PCA

✔ Ancestry inspection

## Association Analysis

✔ Mixed model

✔ GRM

✔ PCs as covariates

## Visualization

✔ QQ plot

✔ Lambda

✔ Manhattan plot

## Replication

✔ Independent cohort

or

✔ Meta-analysis

### Common Beginner Mistakes

### Mistake 1

Running GWAS without QC.

### Mistake 2

Ignoring population structure.

### Mistake 3

Ignoring relatedness.

### Mistake 4

Using only p-values.

Effect sizes matter.

### Mistake 5

Reporting isolated SNPs without checking LD.

### Mistake 6

Assuming genome-wide significance proves causality.

GWAS identifies association, not causation.

#### What Happens After GWAS?

A significant GWAS hit is only the beginning.

Typical follow-up analyses include:

- Fine mapping
- Colocalization
- eQTL analysis
- TWAS
- Polygenic Risk Scores
- Mendelian Randomization
- Functional annotation

### Complete GWAS Workflow

You have now completed the entire GWAS pipeline:

```text
Raw Genotypes
      ↓
Quality Control
      ↓
LD Pruning
      ↓
PCA
      ↓
GRM Construction
      ↓
Mixed Model GWAS
      ↓
QQ Plot
      ↓
Manhattan Plot
      ↓
Meta-analysis
      ↓
Relatedness Inspection
      ↓
Biological Interpretation
```



A GWAS is much more than running a regression for millions of SNPs.

Every stage exists for a reason:

- QC prevents technical artifacts.
- PCA controls ancestry differences.
- GRMs model genetic similarity.
- Mixed models account for relatedness.
- QQ plots diagnose inflation.
- Manhattan plots reveal genomic loci.
- Meta-analysis increases power.

When all of these pieces work together, GWAS becomes one of the most powerful tools in modern human genetics, enabling the discovery of thousands of genetic variants associated with disease, behavior, physiology, and molecular traits.

